# ComLab V3 Emergency Egress Simulation

Agent-based micro-simulation based on the revised proposal for UMTC Visayan Campus ComLab V3. The model includes students, instructor, presiding assistants, lab custodians, locker detours, smoke slowdown, trip events, outward-swinging doors, hallway backpressure, and two staircase directions.

## 1. Setup

In Google Colab, upload `comlab_v3_sim.py` beside this notebook before running the cells.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists('comlab_v3_sim.py'):
    from google.colab import files
    uploaded = files.upload()

from comlab_v3_sim import (
    ComLabV3Simulation,
    ScenarioConfig,
    default_scenarios,
    run_scenarios,
    summarize_results,
)

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Run Baseline and Intervention Scenarios

In [ ]:
configs = default_scenarios(seed=21)
results = run_scenarios(configs, replications=30)
df = pd.DataFrame(summarize_results(results))
df.head()

In [ ]:
summary = (
    df.groupby('scenario')
    .agg(
        evacuation_time_mean=('evacuation_time_s', 'mean'),
        evacuation_time_sd=('evacuation_time_s', 'std'),
        student_clearance_mean=('student_clearance_time_s', 'mean'),
        trips_mean=('trips', 'mean'),
        door_collisions_mean=('door_collisions', 'mean'),
        max_density_mean=('max_cell_density_visits', 'mean'),
        front_door_usage_mean=('front_door_usage', 'mean'),
        back_door_usage_mean=('back_door_usage', 'mean'),
    )
    .round(2)
    .sort_values('student_clearance_mean')
)
summary

## 3. Compare Egress Times

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ordered = summary.index.tolist()
ax.bar(ordered, summary.loc[ordered, 'student_clearance_mean'], color='#2f6f73')
ax.set_ylabel('Mean student clearance time (seconds)')
ax.set_xlabel('Scenario')
ax.set_title('ComLab V3 Evacuation Scenario Comparison')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## 4. Bottleneck Heatmap

Darker cells indicate more accumulated agent presence over the simulation. Use this to discuss center-aisle congestion, locker cross-traffic, and door-threshold bottlenecks.

In [ ]:
selected = ComLabV3Simulation(default_scenarios(seed=42)[1]).run()

fig, ax = plt.subplots(figsize=(7, 8))
im = ax.imshow(selected.density_heatmap, cmap='magma')
ax.scatter([8, 8], [2, 10], marker='s', s=180, c=['#38bdf8', '#22c55e'], label='Doors')
ax.scatter([8], [9], marker='D', s=120, c='#facc15', label='Locker')
ax.scatter([7], [1], marker='P', s=140, c='#ef4444', label='Front extinguisher')
ax.set_title('Panicked Electrical Fire: Density Heatmap')
ax.set_xlabel('Grid X')
ax.set_ylabel('Grid Y')
ax.legend(loc='upper right')
fig.colorbar(im, ax=ax, label='Accumulated agent visits')
plt.tight_layout()
plt.show()

selected.summary()

## 5. Event Log Inspection

In [ ]:
events = pd.DataFrame(selected.event_log)
events.head(20)

In [ ]:
events['event'].value_counts()

## 6. Custom Experiment

Modify the parameters below to test additional proposal interventions.

In [ ]:
custom_config = ScenarioConfig(
    name='custom_reduced_backpressure_no_locker',
    panic=True,
    hazard='electrical_fire',
    locker_enabled=False,
    assigned_exits=True,
    custodians_manage_doors=True,
    hallway_backpressure=0.15,
    v1_flow_intensity=0.35,
    v2_flow_intensity=0.20,
    random_seed=99,
)

custom_results = run_scenarios([custom_config], replications=30)
pd.DataFrame(summarize_results(custom_results)).describe().round(2)